In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [7]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

# model = ChatOllama(model="gemma4:e2b")

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [10]:
from pprint import pprint

pprint(response['messages'][-1].content)

('The `langchain-mcp-adapters` library acts as a bridge between '
 "LangChain/LangGraph and Anthropic's Model Context Protocol (MCP). It "
 'simplifies the integration of MCP tools into LangChain applications by '
 'converting MCP tools into a format compatible with LangChain. This allows '
 'developers to leverage existing MCP servers without the need for manual tool '
 'integration or custom adapters.\n'
 '\n'
 'Key features and benefits include:\n'
 '\n'
 '*   **Seamless Integration:** Easily connect LangChain and LangGraph with '
 'MCP tool servers.\n'
 '*   **Broad Compatibility:** Enables agents to utilize tools from multiple '
 'MCP servers simultaneously.\n'
 '*   **Simplified Development:** Eliminates the complexity of custom adapters '
 'for tool integration.\n'
 '*   **Content Conversion:** Converts multi-part content (like text and '
 "images) from MCP servers into LangChain's standard content blocks.\n"
 '*   **Access to Ecosystem:** Taps into the growing ecosystem of MCP 

## Online MCP

In [11]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [12]:
agent = create_agent(
    model=model,
    tools=tools,
)

In [15]:
question = HumanMessage(content="What time is it in Dhaka?")

response = await agent.ainvoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In [17]:
print(response)

{'messages': [HumanMessage(content='What time is it in Dhaka?', additional_kwargs={}, response_metadata={}, id='fa894764-3550-4662-a2c8-fb0d6b71f0ad'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Dhaka"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dcad3-89ad-7692-9150-4218589633c8-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Dhaka'}, 'id': 'c486b748-dddf-4910-b29a-b7e7134ba956', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 252, 'output_tokens': 20, 'total_tokens': 272, 'input_token_details': {'cache_read': 0}}), ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "Asia/Dhaka",\n  "datetime": "2026-04-26T23:25:55+06:00",\n  "day_of_week": "Sunday",\n  "is_dst": false\n}', 'id': 'lc_4ddcee22-1e13-4cef-9439-ad39a9e6daaa'}], name='